# PrxteinMPNN Training Example

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maraxen/PrxteinMPNN/blob/main/examples/training_example_notebook.ipynb)

This notebook demonstrates how to train PrxteinMPNN with various configurations:

- **Training Modes**: Autoregressive or Diffusion
- **Physics Features**: Electrostatics and/or Van der Waals
- **Precision**: FP32, FP16, or BF16
- **Data Sources**: Example data (fast) or full HuggingFace dataset

Use the form controls below to configure your training run.

## 1. Installation

In [ ]:
# @title Install Dependencies { display-mode: "form" }
# @markdown Run this cell to install PrxteinMPNN and its dependencies.
!uv pip install -q git+https://github.com/maraxen/PrxteinMPNN.git array_record huggingface_hub msgpack msgpack-numpy    

In [ ]:
# @title Import Libraries { display-mode: "form" }

import os
from pathlib import Path

import jax
import jax.numpy as jnp

print(f"JAX version: {jax.__version__}")
print(f"Backend: {jax.default_backend()}")
print(f"Devices: {jax.devices()}")

## 2. Training Configuration

Configure your training run using the form controls below.

In [ ]:
# @title Data Configuration { display-mode: "form" }
# @markdown ### Data Source
# @markdown Choose between example data (fast, for testing) or full dataset.

use_example_data = True  # @param {type: "boolean"}
# @markdown - **True**: Use bundled example data (~1MB, ~50 proteins)
# @markdown - **False**: Download full PDB dataset from HuggingFace (~8GB)

huggingface_repo_id = "maraxen/pdb-2021aug02"  # @param {type: "string"}
# @markdown HuggingFace dataset repository (only used if `use_example_data=False`)

In [ ]:
# @title Training Mode { display-mode: "form" }
# @markdown ### Model Type

training_mode = "autoregressive"  # @param ["autoregressive", "diffusion"]
# @markdown - **autoregressive**: Standard ProteinMPNN training (faster)
# @markdown - **diffusion**: Discrete diffusion model (experimental)

In [ ]:
# @title Physics Features { display-mode: "form" }
# @markdown ### Physics-Augmented Training
# @markdown Enable physics-based features for enhanced model performance.

use_electrostatics = False  # @param {type: "boolean"}
# @markdown Electrostatic force projections (requires PQR data with charges)

use_vdw = False  # @param {type: "boolean"}
# @markdown Van der Waals force projections (requires LJ parameters)

In [ ]:
# @title Hyperparameters { display-mode: "form" }
# @markdown ### Optimization Settings

learning_rate = 1e-4  # @param {type: "number"}
weight_decay = 0.01  # @param {type: "number"}
batch_size = 8  # @param {type: "integer"}
num_epochs = 3  # @param {type: "integer"}
warmup_steps = 100  # @param {type: "integer"}

# @markdown ### Regularization
label_smoothing = 0.1  # @param {type: "slider", min: 0, max: 0.3, step: 0.05}
gradient_clip = 2.0  # @param {type: "number"}

# @markdown ### Precision
precision = "bf16"  # @param ["fp32", "fp16", "bf16"]
# @markdown - **bf16**: Recommended for TPU/modern GPU (fastest)
# @markdown - **fp16**: Good for older GPUs
# @markdown - **fp32**: Full precision (slowest, most accurate)

In [ ]:
# @title Diffusion Settings { display-mode: "form" }
# @markdown ### Diffusion-Specific Parameters
# @markdown *Only used when `training_mode="diffusion"`*

diffusion_num_steps = 1000  # @param {type: "integer"}
diffusion_schedule_type = "cosine"  # @param ["cosine", "linear"]
diffusion_beta_start = 1e-4  # @param {type: "number"}
diffusion_beta_end = 0.02  # @param {type: "number"}

In [ ]:
# @title Checkpointing & Early Stopping { display-mode: "form" }
# @markdown ### Checkpoint Settings

checkpoint_dir = "./checkpoints"  # @param {type: "string"}
checkpoint_every = 500  # @param {type: "integer"}
keep_last_n_checkpoints = 3  # @param {type: "integer"}

# @markdown ### Early Stopping
use_early_stopping = False  # @param {type: "boolean"}
early_stopping_patience = 5  # @param {type: "integer"}
early_stopping_metric = "val_loss"  # @param ["val_loss", "val_accuracy", "val_perplexity"]

# @markdown ### Logging
log_every = 50  # @param {type: "integer"}
eval_every = 200  # @param {type: "integer"}

## 3. Data Loading

In [ ]:
# @title Load Training Data { display-mode: "form" }

import shutil

def get_data_paths(use_example: bool, hf_repo: str) -> tuple[Path, Path]:
    """Get paths to training data based on configuration.
    
    Returns:
        Tuple of (array_record_path, index_path)
    """
    if use_example:
        # Use bundled example data
        # Try multiple possible locations
        possible_paths = [
            Path("data/pdb_sample.array_record"),  # Repo root
            Path("../data/pdb_sample.array_record"),  # From examples/
            Path("/content/PrxteinMPNN/data/pdb_sample.array_record"),  # Colab clone
        ]
        
        for p in possible_paths:
            if p.exists():
                print(f"✓ Found example data at: {p}")
                index_path = p.with_suffix(".index.json")
                return p, index_path
        
        # If not found, try to clone the repo (for Colab)
        print("Example data not found locally. Cloning repository...")
        import subprocess
        subprocess.run(["git", "clone", "--depth=1", 
                       "https://github.com/maraxen/PrxteinMPNN.git", 
                       "/content/PrxteinMPNN"], check=True)
        p = Path("/content/PrxteinMPNN/data/pdb_sample.array_record")
        print(f"✓ Cloned repo, using data at: {p}")
        return p, p.with_suffix(".index.json")
    
    else:
        # Download from HuggingFace
        from huggingface_hub import hf_hub_download
        
        print(f"Downloading full dataset from HuggingFace: {hf_repo}")
        print("This may take several minutes...")
        
        cache_dir = Path("./hf_cache")
        cache_dir.mkdir(exist_ok=True)
        
        try:
            record_path = hf_hub_download(
                repo_id=hf_repo,
                filename="pdb_2021aug02.array_record",
                repo_type="dataset",
                local_dir=cache_dir,
                local_dir_use_symlinks=False,
            )
            
            index_path = hf_hub_download(
                repo_id=hf_repo,
                filename="index.json",
                repo_type="dataset",
                local_dir=cache_dir,
                local_dir_use_symlinks=False,
            )
            
            print(f"✓ Downloaded dataset to: {cache_dir}")
            return Path(record_path), Path(index_path)
            
        except Exception as e:
            print(f"⚠ Failed to download from HuggingFace: {e}")
            print("Falling back to example data...")
            return get_data_paths(True, hf_repo)


# Load data based on configuration
array_record_path, index_path = get_data_paths(use_example_data, huggingface_repo_id)

print(f"\nData configuration:")
print(f"  ArrayRecord: {array_record_path}")
print(f"  Index: {index_path}")
print(f"  File size: {array_record_path.stat().st_size / 1024 / 1024:.1f} MB")

## 4. Training

In [ ]:
# @title Build Training Specification { display-mode: "form" }

from prxteinmpnn.training.specs import TrainingSpecification

# Build the training specification from form inputs
spec = TrainingSpecification(
    # Data
    inputs=str(array_record_path),
    use_preprocessed=True,
    preprocessed_index_path=str(index_path) if index_path.exists() else None,
    
    # Training mode
    training_mode=training_mode,
    
    # Physics features
    use_electrostatics=use_electrostatics,
    use_vdw=use_vdw,
    
    # Optimization
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    batch_size=batch_size,
    num_epochs=num_epochs,
    warmup_steps=warmup_steps,
    gradient_clip=gradient_clip if gradient_clip > 0 else None,
    label_smoothing=label_smoothing,
    
    # Precision
    precision=precision,
    
    # Diffusion settings (only used if training_mode="diffusion")
    diffusion_num_steps=diffusion_num_steps,
    diffusion_schedule_type=diffusion_schedule_type,
    diffusion_beta_start=diffusion_beta_start,
    diffusion_beta_end=diffusion_beta_end,
    
    # Checkpointing
    checkpoint_dir=checkpoint_dir,
    checkpoint_every=checkpoint_every,
    keep_last_n_checkpoints=keep_last_n_checkpoints,
    
    # Early stopping
    early_stopping_patience=early_stopping_patience if use_early_stopping else None,
    early_stopping_metric=early_stopping_metric,
    
    # Logging
    log_every=log_every,
    eval_every=eval_every,
)

print("Training Specification:")
print("=" * 50)
print(f"  Mode: {spec.training_mode}")
print(f"  Batch size: {spec.batch_size}")
print(f"  Learning rate: {spec.learning_rate}")
print(f"  Epochs: {spec.num_epochs}")
print(f"  Precision: {spec.precision}")
print(f"  Physics: EStat={spec.use_electrostatics}, VdW={spec.use_vdw}")
if spec.training_mode == "diffusion":
    print(f"  Diffusion steps: {spec.diffusion_num_steps}")
    print(f"  Schedule: {spec.diffusion_schedule_type}")
print("=" * 50)

In [ ]:
# @title Run Training { display-mode: "form" }
# @markdown Click **Run** to start training with the configured settings.

from prxteinmpnn.training.trainer import train

print("Starting training...")
print(f"Checkpoint directory: {spec.checkpoint_dir}")
print()

# Run training
result = train(spec)

print()
print("=" * 50)
print("Training Complete!")
print("=" * 50)
print(f"Final step: {result.final_step}")
print(f"Checkpoints saved to: {result.checkpoint_dir}")

## 5. Results & Visualization

In [ ]:
# @title List Checkpoints { display-mode: "form" }

checkpoint_path = Path(checkpoint_dir)
if checkpoint_path.exists():
    checkpoints = sorted(checkpoint_path.glob("*.eqx"))
    print(f"Found {len(checkpoints)} checkpoint(s):")
    for ckpt in checkpoints:
        size_mb = ckpt.stat().st_size / 1024 / 1024
        print(f"  - {ckpt.name} ({size_mb:.1f} MB)")
else:
    print(f"No checkpoints found at {checkpoint_dir}")

In [ ]:
# @title Load Trained Model { display-mode: "form" }
# @markdown Load the trained model for inference or continued training.

checkpoint_to_load = "latest"  # @param {type: "string"}
# @markdown Enter "latest" to load the most recent checkpoint, or specify a filename.

import equinox as eqx
from prxteinmpnn.model.mpnn import PrxteinMPNN

checkpoint_path = Path(checkpoint_dir)

if checkpoint_to_load == "latest":
    checkpoints = sorted(checkpoint_path.glob("*.eqx"))
    if checkpoints:
        ckpt_file = checkpoints[-1]
    else:
        raise FileNotFoundError("No checkpoints found")
else:
    ckpt_file = checkpoint_path / checkpoint_to_load

print(f"Loading checkpoint: {ckpt_file}")

# Load the model
trained_model = eqx.tree_deserialise_leaves(ckpt_file, result.final_model)
print("✓ Model loaded successfully!")

# Print model info
num_params = sum(x.size for x in jax.tree_util.tree_leaves(eqx.filter(trained_model, eqx.is_array)))
print(f"  Parameters: {num_params:,}")

## 6. Next Steps

After training, you can:

1. **Use for inference**: Load the trained model and use it for sequence design
2. **Continue training**: Set `resume_from_checkpoint` in a new `TrainingSpecification`
3. **Evaluate**: Run validation on held-out data
4. **Export**: Save the model for deployment

### Example: Using the trained model for sampling

```python
from prxteinmpnn.sampling import make_sample_sequences, SamplingConfig
from prxteinmpnn.io import from_structure_file, protein_structure_to_model_inputs

# Load a structure
protein = from_structure_file("my_protein.pdb")
inputs = protein_structure_to_model_inputs(protein)

# Create sampler with trained model
sample_fn = make_sample_sequences(
    trained_model,
    decoding_order_fn=random_decoding_order,
    config=SamplingConfig(temperature=0.1),
    model_inputs=inputs,
)

# Sample sequences
key = jax.random.PRNGKey(42)
sampled_seq, logits, order = sample_fn(prng_key=key)
```

See the [main example notebook](./example_notebook.ipynb) for more inference examples.